In [ ]:
import numpy as np

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.circuit import Qubit, Clbit, marginal_counts
from qiskit.circuit.library import HGate, XGate, SGate, SdgGate
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.result import marginal_counts
from qiskit.visualization import plot_histogram

In [39]:
# Magic numbers
OPT_LEVEL = 0
SHOTS = 4096
SEED = 7

NOISE = ("ideal", "gates", "readout", "combined")
BASIS_1Q = ['h','x','z','s','sdg','ry','rz','id']
BASIS_2Q = ['cx']
BASIS = BASIS_1Q + BASIS_2Q

LAM1, LAM2 = 0.005, 0.02
P = 0.03

In [40]:
# Helper functions
def add_noise(kind: str):
    # helper to add noise to the AerSimulator
    nm = NoiseModel()
    if kind in ("gates", "combined"):
        nm.add_all_qubit_quantum_error(depolarizing_error(LAM1, 1), BASIS_1Q)
        nm.add_all_qubit_quantum_error(depolarizing_error(LAM2, 2), BASIS_2Q)
    if kind in ("readout", "combined"):
        nm.add_all_qubit_readout_error(ReadoutError([[1 - P, P],[P, 1 - P]]))
    return nm

def run(qc: QuantumCircuit, 
        noise: str, 
        shots: int = SHOTS, 
        seed: int = SEED, 
        opt_level: int = OPT_LEVEL):
    # helper to run the thing
    sim = AerSimulator(method="density_matrix",
                       noise_model= add_noise(noise),
                       basis_gates=BASIS)
    tqc = transpile(qc, sim, optimization_level=opt_level, seed_transpiler=seed)
    result = sim.run(tqc, shots=shots, seed_simulator=seed).result()
    return result, tqc

# Problem A: Quantum Teleportation

In [41]:
PROBES = {
    "0": [],
    "1": [XGate()],
    "+": [HGate()],
    "-": [XGate(), HGate()],
    "+i": [HGate(), SGate()],
    "-i": [HGate(), SdgGate()]
}

# get the preparation and inverse-preparation stuff
def get_prep_circuits(probe):
    prep = QuantumCircuit(1, name=f"prep_probe")
    for gate in PROBES[probe]:
        prep.append(gate, [0])
    post = prep.inverse()
    post.name = f"post_probe"
    return prep.to_gate(), post.to_gate()

def quantum_teleportation(probe, bob_corrects=True):
    # q[0] = qubit to be sent, q[1..2] = e-bits, q[3..5] = clbit
    # bob_corrects is whether bob will correct his qubit
    q = QuantumCircuit(3, 3)

    # prepare qubit
    prep, post = get_prep_circuits(probe)
    q.append(prep, [0])
    q.barrier()

    # create entangled pair between q[1] and q[2]
    q.h(1)
    q.cx(1, 2)

    # entangle "unknown" probe state
    q.cx(0, 1)

    # measure on alice side
    q.h(0)
    q.measure(0, 0)
    q.measure(1, 1)

    # bob corrects
    if bob_corrects:
        with q.if_test((q.clbits[0], 1)):
            q.z(2)
        with q.if_test((q.clbits[1], 1)):
            q.x(2)

    # post-processing + measure
    q.barrier()
    q.append(post, [2])
    q.measure(2, 2)
    return q

## Success Probabilities
Over all noise configs and probes, and whether Bob corrects

In [53]:
for corrected in (True, False):
    print(f"Corrected: {corrected}")
    for noise in NOISE:
        res = {}
        for probe in PROBES:
            qt = quantum_teleportation(probe,  corrected)
            result, tqc = run(qt, noise)
            result_dict = result.get_counts()
            res[probe] = round(marginal_counts(result_dict, [2])["0"] / 4096, 4)
        print(f"- {noise}: {res}")
    print(f"{'=' * 10}")

Corrected: True
- ideal: {'0': 1.0, '1': 1.0, '+': 1.0, '-': 1.0, '+i': 1.0, '-i': 1.0}
- gates: {'0': 0.9766, '1': 0.9719, '+': 0.9653, '-': 0.9595, '+i': 0.9595, '-i': 0.9595}
- readout: {'0': 0.9414, '1': 0.9414, '+': 0.9368, '-': 0.9368, '+i': 0.9099, '-i': 0.9099}
- combined: {'0': 0.9216, '1': 0.9172, '+': 0.9087, '-': 0.905, '+i': 0.8804, '-i': 0.8804}
Corrected: False
- ideal: {'0': 0.501, '1': 0.501, '+': 0.501, '-': 0.501, '+i': 0.501, '-i': 0.501}
- gates: {'0': 0.501, '1': 0.501, '+': 0.501, '-': 0.501, '+i': 0.501, '-i': 0.501}
- readout: {'0': 0.499, '1': 0.499, '+': 0.499, '-': 0.499, '+i': 0.499, '-i': 0.499}
- combined: {'0': 0.499, '1': 0.499, '+': 0.499, '-': 0.499, '+i': 0.499, '-i': 0.499}
